# 04. Final evaluation and error analysis

Uses the **saved predictions** from `02_baselines.ipynb` and `03_model.ipynb`
(`results/predictions/*.csv.gz`), so nothing is retrained and every number matches `results/metrics.csv`.
Run notebooks 02 and 03 first (or `python -m src.pipeline`).

The final CatBoost model is chosen by **validation** MAPE, then compared with `seasonal_naive_last_year`
on the **independent 2016 test** rows (identical rows for both models, checked in code).

In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd()
PROJECT_ROOT = next(
    (p for p in (current_dir, current_dir.parent) if (p / "src" / "validation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(f"Project root not found from: {current_dir}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

RESULTS_DIR = PROJECT_ROOT / "results"
TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image

from src.error_analysis import run_error_analysis
from src.experiments import prepare_data, select_final_model
from src.model import MODEL_REGISTRY
from src.results import check_same_evaluation_rows

metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
FINAL_MODEL = select_final_model(metrics)
print("Final model (lowest validation MAPE):", FINAL_MODEL)

check_same_evaluation_rows(
    RESULTS_DIR,
    ["seasonal_naive_last_year", "group_median", "ridge_calendar_one_hot", *MODEL_REGISTRY],
)
print("All models were evaluated on identical rows in every split.")

## 1. Error tables

In [ ]:
train = prepare_data(TRAIN_PATH)
res = run_error_analysis(RESULTS_DIR, FINAL_MODEL, train_df=train, focus_country="Norway")
tables = res["tables"]
print("Overall test MAPE - model: %.2f%% | seasonal_naive_last_year: %.2f%%"
      % (res["pair"]["ape_model"].mean(), res["pair"]["ape_baseline"].mean()))

### By country (test 2016; `gap > 0` means the model is better than the baseline)

In [ ]:
tables["country"].round(3)

In [ ]:
tables["store"].round(3)

In [ ]:
tables["product"].round(3)

### By year (validation years were used for tuning; only `test` is an independent estimate)

In [ ]:
tables["by_year"].round(3)

### Country x year

In [ ]:
tables["country_year"].round(3)

## 2. Where is the final CatBoost model better / worse than the baselines?

First compare the validation-selected CatBoost model with **all three baselines** on the independent 2016 test.

In [ ]:
baseline_names = ["group_median", "seasonal_naive_last_year", "ridge_calendar_one_hot"]
test_scores = (
    metrics.loc[metrics["validation_scheme"].eq("final_test")]
    .set_index("model")["mape"]
)
final_test_mape = float(test_scores.loc[FINAL_MODEL])
baseline_comparison = pd.DataFrame({
    "test_mape": test_scores.loc[baseline_names],
    "gap_baseline_minus_model": test_scores.loc[baseline_names] - final_test_mape,
})
baseline_comparison["catboost_better"] = baseline_comparison["gap_baseline_minus_model"] > 0
baseline_comparison.round(3)

### Country-level comparison against `seasonal_naive_last_year`

The seasonal-naive model is the strongest baseline on the independent test, so the country-level error analysis below uses it as the reference.

In [ ]:
c = tables["country"]
better = c.loc[c["model_better"], "country"].tolist()
worse = c.loc[~c["model_better"], "country"].tolist()
print("Model better than baseline in:", better)
print("Model worse than baseline in: ", worse)

## 3. Norway

The old long-horizon explanation must **not** be reused. Under the old 2014-2016 evaluation trained only on 2010-2013, `seasonal_naive_last_year` could not observe 2014/2015 test targets and therefore could not really "react one year later" to them.

Under the corrected scheme every evaluation window is one year long:

- for validation 2015, training ends in 2014, so the 2015 level shift is unseen by both models;
- for the independent 2016 test, training ends in 2015, so the seasonal baseline can use actual 2015 sales as the previous-year value when available.

The table below shows the mean actual level and signed percentage bias for Norway by product/year. Use it to verify the level-shift interpretation instead of assuming model behaviour.

In [ ]:
res["focus_table"].round(2)

In [ ]:
no = res["pair_all"][res["pair_all"]["country"] == "Norway"]
yearly = no.groupby("year")[["y_true", "pred_model", "pred_baseline"]].mean()
yearly.plot(marker="o", figsize=(7, 3.5), title="Norway: mean daily sales (evaluated years)")
plt.tight_layout(); plt.show()

## 4. Worst series

In [ ]:
res["worst"].round(2)

In [ ]:
Image(filename=str(RESULTS_DIR / "error_analysis_worst_series.png"))

## 5. Conclusion

In [ ]:
display(Markdown(res["summary"]))

In [ ]:
country_table = tables["country"]
country_year = tables["country_year"]
norway_test = country_table.loc[country_table["country"].eq("Norway")].iloc[0]
norway_2015 = country_year.loc[(country_year["country"].eq("Norway")) & (country_year["year"].eq(2015))].iloc[0]
norway_2016 = country_year.loc[(country_year["country"].eq("Norway")) & (country_year["year"].eq(2016))].iloc[0]

val_means = (
    metrics.loc[metrics["validation_scheme"].eq("expanding_window")]
    .groupby("model")["mape"].mean()
)
plain_log = float(val_means.loc["catboost_log"])
tuned_log = float(val_means.loc["catboost_log_tuned"])

all_baselines_beaten = bool(baseline_comparison["catboost_better"].all())
all_countries_beaten = bool(country_table["model_better"].all())
model_col = "mape_" + FINAL_MODEL

conclusion = f"""### Final conclusion

- **Model selection:** `{FINAL_MODEL}` is selected only by mean validation MAPE ({tuned_log:.3f}%). For the log-target CatBoost, tuning is practically negligible compared with the untuned log model ({plain_log:.3f}%).
- **Independent test:** `{FINAL_MODEL}` reaches **{final_test_mape:.2f}% MAPE** on 2016. It beats all three baselines on the same test rows: {'yes' if all_baselines_beaten else 'no'}.
- **Country-level errors:** against the strongest baseline, `seasonal_naive_last_year`, CatBoost is better in every country: {'yes' if all_countries_beaten else 'no'}. Norway is the hardest country for CatBoost ({norway_test[model_col]:.2f}% vs {norway_test['mape_seasonal_naive_last_year']:.2f}% for the baseline).
- **Norway level shift:** in validation 2015, after the sales level drops, both methods have very large errors ({norway_2015[model_col]:.2f}% CatBoost vs {norway_2015['mape_seasonal_naive_last_year']:.2f}% baseline), because 2015 targets are not available to either model. In the independent 2016 test, training includes 2015; errors fall to {norway_2016[model_col]:.2f}% and {norway_2016['mape_seasonal_naive_last_year']:.2f}% respectively. This is the correct setting in which the seasonal baseline can use the lower 2015 values as last-year observations.
- **Interpretation boundary:** validation-year error tables are diagnostic but were involved in model selection; only the 2016 test is an independent estimate of final performance.
"""
display(Markdown(conclusion))

## 6. Reproducibility check

The clean-environment verification is recorded in `results/reproducibility_check.md`.

In [ ]:
repro_path = RESULTS_DIR / "reproducibility_check.md"
if repro_path.exists():
    display(Markdown(repro_path.read_text(encoding="utf-8")))
else:
    print("Reproducibility record is missing. Run: bash scripts/clean_env_check.sh data/train.csv")